# Germany Hourly Power Generation — EDA, Seasonality & Thermal Forecasting

**Purpose.** A readable, editable end-to-end template for hourly German power-generation data.

**Important data note.** The accompanying CSV files are **synthetic teaching data**, not official German grid data. They reproduce the requested structure and broad relationships so every cell can run immediately. Replace the CSV path with the real dataset when available.

## Project map
1. Data structure, cleaning and aggregation
2. EDA by source subcategory
3. Hypothesis tests: hour-of-day, month and weekday
4. STL decomposition
5. Daily Solar distribution and normalizing transformations
6. Thermal prediction: time split, ~200 candidate features, screening, model comparison and ensemble

## Key forecasting assumption
For the prediction exercise, contemporaneous Wind and Solar observations are allowed, while **all other generation categories are restricted to lag-1**. This means the task is a conditional nowcast if Wind/Solar at time *t* are known at prediction time; it is not a pure one-hour-ahead forecast unless those inputs are forecast or otherwise available in advance.

## 0. Install / import packages

Core dependencies: `pandas`, `numpy`, `matplotlib`, `scipy`, `statsmodels`, `scikit-learn`.

In [ ]:
# If needed, uncomment:
# %pip install pandas numpy matplotlib scipy statsmodels scikit-learn

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import skew, kurtosis
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import jarque_bera

from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
    HistGradientBoostingRegressor, GradientBoostingRegressor, AdaBoostRegressor)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)

## 1. Data structure assumptions

Assume the raw file is in **long format**:

| timestamp | category | subcategory | generation_mw |
|---|---|---|---:|
| 2023-01-01 00:00 | Thermal | Lignite | 12345 |
| 2023-01-01 00:00 | Wind | Onshore Wind | 6789 |

- One row = one timestamp × one source subcategory.
- `generation_mw` is hourly average/instantaneous generation in MW (the actual data dictionary must confirm which).
- Categories: Thermal, Water, Wind, Nuclear, Solar.
- Thermal target = sum of all Thermal subcategories at each hour.
- Wind/Solar predictors = sum of their subcategories at each hour.
- `Pumped Storage` is included as a Water subcategory for the storage-arbitrage discussion.

In [ ]:
from pathlib import Path

DATA_PATH = Path("germany_hourly_generation_long.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("germany_power_generation_project") / DATA_PATH

raw = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
raw = raw.sort_values(["timestamp", "category", "subcategory"]).reset_index(drop=True)

print("Shape:", raw.shape)
display(raw.head())
display(raw.dtypes)


### 1.1 Basic data checks and missing values

Check duplicated keys, chronological order, missing timestamps and missing generation values. Do not silently forward-fill long gaps. For this teaching example, interpolate short gaps within each subcategory and then use time interpolation as a fallback.

In [ ]:
key_cols = ["timestamp", "category", "subcategory"]

print("Duplicate keys:", raw.duplicated(key_cols).sum())
print("Missing generation ratio:", raw["generation_mw"].isna().mean())
print("Timestamp range:", raw["timestamp"].min(), "to", raw["timestamp"].max())

# Average accidental duplicate records.
raw = raw.groupby(key_cols, as_index=False)["generation_mw"].mean()

# Fill only short gaps within each subcategory; keep long gaps visible.
parts = []
for _, g in raw.groupby(["category", "subcategory"], sort=False):
    g = g.sort_values("timestamp").set_index("timestamp")
    g["generation_mw"] = g["generation_mw"].interpolate(
        method="time", limit=3, limit_area="inside"
    )
    parts.append(g.reset_index())

raw = pd.concat(parts, ignore_index=True)
display(raw.isna().mean().rename("missing_ratio").to_frame())


### 1.2 Aggregate subcategories to category-level hourly series

Use `pivot_table` to create one column per subcategory, then sum subcategories within each main category.

In [ ]:
subcategory_wide = raw.pivot_table(
    index="timestamp",
    columns=["category", "subcategory"],
    values="generation_mw",
    aggfunc="sum"
).sort_index()

category_hourly = (
    raw.groupby(["timestamp", "category"])["generation_mw"]
       .sum()
       .unstack("category")
       .sort_index()
)

# Restore missing hours and fill only short internal gaps.
full_index = pd.date_range(
    category_hourly.index.min(),
    category_hourly.index.max(),
    freq="h"
)
category_hourly = category_hourly.reindex(full_index)
category_hourly.index.name = "timestamp"
category_hourly = category_hourly.interpolate(
    method="time", limit=3, limit_area="inside"
)

display(category_hourly.head())
display(category_hourly.isna().sum().rename("missing_hours").to_frame())


## 2. EDA by source subcategory

### 2.1 Mean generation, volatility and relative volatility

- Mean: average generation level.
- Standard deviation: absolute volatility.
- Coefficient of variation (CV = std / mean): relative volatility, useful when series have different scales.

In [ ]:
subcategory_stats = (
    raw.groupby(["category", "subcategory"])["generation_mw"]
       .agg(mean="mean", std="std", minimum="min", maximum="max")
       .reset_index()
)
subcategory_stats["cv"] = subcategory_stats["std"] / subcategory_stats["mean"]
display(subcategory_stats.sort_values(["category", "mean"], ascending=[True, False]))

### 2.2 Time-series plots and moving averages

Plot the hourly series and 24-hour / 7-day moving averages. The moving average smooths short-run noise and helps reveal slower trends.

In [ ]:
# Category-level trends
cols = [c for c in ["Thermal", "Water", "Wind", "Solar", "Nuclear"] if c in category_hourly]
fig, axes = plt.subplots(len(cols), 1, figsize=(14, 2.5 * len(cols)), sharex=True)
axes = np.atleast_1d(axes)
for ax, col in zip(axes, cols):
    s = category_hourly[col]
    ax.plot(s.index, s, alpha=0.35, linewidth=0.6, label="Hourly")
    ax.plot(s.index, s.rolling(24, min_periods=12).mean(), label="24h MA")
    ax.plot(s.index, s.rolling(168, min_periods=84).mean(), label="7d MA")
    ax.set_title(f"{col}: hourly generation")
    ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Subcategory trends: plot the largest subcategories by average generation.
top_subcats = (
    raw.groupby(["category", "subcategory"])["generation_mw"]
       .mean().sort_values(ascending=False).head(8).index
)
fig, axes = plt.subplots(len(top_subcats), 1, figsize=(14, 2.2 * len(top_subcats)), sharex=True)
axes = np.atleast_1d(axes)
for ax, key in zip(axes, top_subcats):
    s = subcategory_wide[key]
    ax.plot(s.index, s, alpha=0.35, linewidth=0.6)
    ax.plot(s.index, s.rolling(24, min_periods=12).mean(), linewidth=1)
    ax.set_title(f"{key[0]} — {key[1]}")
plt.tight_layout()
plt.show()


### 2.3 Autocorrelation, partial autocorrelation and ramp behaviour

- **ACF**: correlation between a series and its lagged values. Use it to identify persistence and repeating lags (24h, 168h).
- **PACF**: correlation at a lag after controlling for shorter lags.
- **Cross-correlation**: inspect contemporaneous and lagged relationships between generation categories. Correlation is descriptive, not causal.
- **Ramp rate**: hourly change in generation, useful for identifying fast system adjustments.

Use these plots to motivate feature windows; do not choose features from the test period.

ACF/PACF on raw levels can be inflated by trend and seasonality. Read peaks at 24 and 168 as possible daily/weekly persistence, then confirm with plots or differenced series. Lagged cross-correlation is descriptive and does not establish causality. The ANOVA/Kruskal p-values in the next section are also descriptive because hourly observations are serially dependent.

In [ ]:
cols = [c for c in ["Thermal", "Wind", "Solar", "Water"] if c in category_hourly]
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, col in zip(axes.flat, cols):
    s = category_hourly[col].dropna()
    plot_acf(s, lags=min(168, len(s) - 1), ax=ax, title=f"{col}: ACF")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
for ax, col in zip(axes.flat, cols):
    s = category_hourly[col].dropna()
    plot_pacf(s, lags=min(48, max(1, len(s)//2 - 1)), method="ywm", ax=ax, title=f"{col}: PACF")
plt.tight_layout()
plt.show()

# Contemporaneous category correlations (descriptive, not causal).
corr = category_hourly[["Thermal", "Wind", "Solar", "Water", "Nuclear"]].corr()
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=30, ha="right")
ax.set_yticks(range(len(corr.index)), corr.index)
for r in range(len(corr)):
    for c in range(len(corr)):
        ax.text(c, r, f"{corr.iloc[r, c]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax, label="Pearson correlation")
ax.set_title("Hourly generation correlation")
plt.tight_layout()
plt.show()

# Ramp rate = hour-to-hour change.
ramp = category_hourly[["Thermal", "Wind", "Solar", "Water", "Nuclear"]].diff()
display(ramp.agg(["mean", "std", "min", "max"]).T)


# Lagged cross-correlation: past Wind/Solar against current Thermal.
# Positive lag k means comparing Thermal(t) with renewable(t-k).
fig, ax = plt.subplots(figsize=(10, 4))
for col in ["Wind", "Solar"]:
    values = [
        category_hourly["Thermal"].corr(category_hourly[col].shift(k))
        for k in range(49)
    ]
    ax.plot(range(49), values, marker="o", markersize=2, label=col)
ax.axhline(0, color="black", linewidth=0.7)
ax.set_xlabel("Lag (hours)")
ax.set_ylabel("Correlation with current Thermal")
ax.set_title("Lagged cross-correlation: renewable generation vs Thermal")
ax.legend()
plt.tight_layout()
plt.show()


### 2.4 Hypothesis tests: intraday, monthly and weekday effects

Use the time-series plots to motivate hypotheses rather than testing every pattern blindly.

For each grouping, the null is that the group means are equal:
- Intraday: all 24 hour-of-day means are equal.
- Monthly: all 12 month means are equal.
- Weekday: all 7 weekday means are equal.

One-way ANOVA is a mean-comparison test. Kruskal–Wallis is a rank-based alternative when distributions are strongly non-normal or contain outliers. These tests do not establish causality.

In [ ]:
def add_time_fields(df):
    out = df.copy()
    out["hour"] = out.index.hour
    out["month"] = out.index.month
    out["weekday"] = out.index.dayofweek
    return out

def grouped_mean_test(series, group_labels, test_name="ANOVA"):
    temp = pd.DataFrame({"y": series, "group": group_labels}).dropna()
    groups = [g["y"].to_numpy() for _, g in temp.groupby("group")]
    if len(groups) < 2:
        return {"test": test_name, "stat": np.nan, "p_value": np.nan}
    if test_name == "ANOVA":
        stat, p = stats.f_oneway(*groups)
    else:
        stat, p = stats.kruskal(*groups)
    return {"test": test_name, "stat": stat, "p_value": p}

def seasonality_test_table(df, columns):
    rows = []
    for col in columns:
        s = df[col].dropna()
        for label, groups in [
            ("hour_of_day", s.index.hour),
            ("month", s.index.month),
            ("weekday", s.index.dayofweek),
        ]:
            groups = pd.Series(groups, index=s.index)
            for test in ["ANOVA", "Kruskal"]:
                result = grouped_mean_test(s, groups, test)
                rows.append({"series": col, "grouping": label, **result})
    return pd.DataFrame(rows)

test_results = seasonality_test_table(category_hourly, category_hourly.columns)
display(test_results)

### 2.5 Targeted hypotheses based on the prompt

Suggested hypotheses to check after viewing the plots:

| Series | Pattern to investigate | Grouping |
|---|---|---|
| Solar | Strong intraday profile; seasonal/monthly differences | hour, month |
| Wind | Intraday and seasonal/monthly variation | hour, month |
| Thermal | Possible weekday/weekend operating pattern | weekday |
| Nuclear | Relatively stable output | stability / variability metrics |

For Solar, a raw hourly ANOVA across hours will almost certainly detect the day-night pattern. For Nuclear, a non-significant test is not proof of stability: also inspect CV, range and rolling standard deviation.

In [ ]:
# Solar: intraday and monthly effects
for grouping, labels in [
    ("hour", category_hourly.index.hour),
    ("month", category_hourly.index.month)
]:
    print("Solar:", grouping)
    print(grouped_mean_test(category_hourly["Solar"], pd.Series(labels, index=category_hourly.index), "ANOVA"))

# Wind: intraday and monthly effects
for grouping, labels in [
    ("hour", category_hourly.index.hour),
    ("month", category_hourly.index.month)
]:
    print("Wind:", grouping)
    print(grouped_mean_test(category_hourly["Wind"], pd.Series(labels, index=category_hourly.index), "ANOVA"))

# Thermal: weekday effect
print("Thermal weekday effect:")
print(grouped_mean_test(
    category_hourly["Thermal"],
    pd.Series(category_hourly.index.dayofweek, index=category_hourly.index),
    "ANOVA"
))

# Nuclear stability diagnostics
nuclear = category_hourly["Nuclear"].dropna()
print({
    "mean": nuclear.mean(),
    "std": nuclear.std(),
    "CV": nuclear.std() / nuclear.mean(),
    "min": nuclear.min(),
    "max": nuclear.max(),
    "range_pct_of_mean": (nuclear.max() - nuclear.min()) / nuclear.mean()
})

### 2.6 STL decomposition

For an additive decomposition:

\[
Y_t = T_t + S_t + R_t
\]

where \(T_t\) is trend, \(S_t\) is seasonal component, and \(R_t\) is residual.

For a multiplicative decomposition:

\[
Y_t = T_t \times S_t \times R_t
\]

A log transform can convert a multiplicative relationship into an additive one when all values are positive.

For hourly data, `period=24` extracts the daily cycle; `period=168` extracts a weekly cycle. STL is applied to one seasonal period at a time.

In [ ]:
# Example: daily seasonality in hourly Solar
solar_stl = STL(category_hourly["Solar"].dropna(), period=24, robust=True).fit()
solar_stl.plot()
plt.tight_layout()
plt.show()

# Example: weekly seasonality in hourly Thermal
thermal_stl_week = STL(category_hourly["Thermal"].dropna(), period=168, robust=True).fit()
thermal_stl_week.plot()
plt.tight_layout()
plt.show()

## 3. Solar daily analysis

### 3.1 Aggregate hourly Solar generation to daily data

For hourly average power in MW, summing 24 hourly MW observations is not energy in MWh. If the source values are hourly average MW, daily energy is `sum(MW) × 1 hour`, numerically the sum in MWh. If the source is already MWh per interval, sum directly. Here we use daily sum as an interval-energy proxy and label it clearly.

In [ ]:
solar_daily = category_hourly["Solar"].resample("D").sum(min_count=1)
solar_daily.name = "solar_daily_generation_mwh_proxy"
display(solar_daily.head())

### 3.2 Daily time series and moving averages

Daily aggregation removes intraday variation. This plot can show trend and month/season differences, but it cannot show the within-day solar cycle.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(solar_daily.index, solar_daily, alpha=0.45, label="Daily")
ax.plot(solar_daily.index, solar_daily.rolling(30, min_periods=15).mean(), label="30-day MA")
ax.plot(solar_daily.index, solar_daily.rolling(90, min_periods=45).mean(), label="90-day MA")
ax.set_title("Daily Solar Generation")
ax.set_ylabel("Daily generation (MWh proxy)")
ax.legend()
plt.tight_layout()
plt.show()

### 3.3 Distribution: histogram, skewness, kurtosis, Q-Q plot and normality tests

- Skewness near 0 indicates approximate symmetry.
- Excess kurtosis near 0 matches the normal distribution's tail thickness.
- Q-Q points close to the diagonal indicate approximate normality.
- With large samples, formal tests can reject tiny deviations; combine test results with plots and effect size.

In [ ]:
def distribution_report(series, title="Series"):
    x = pd.Series(series).dropna()
    print(f"{title}: n={len(x)}")
    print("Mean:", x.mean(), "Std:", x.std())
    print("Skewness:", skew(x, bias=False))
    print("Excess kurtosis:", kurtosis(x, fisher=True, bias=False))

    # D'Agostino-Pearson normality test requires at least 8 observations.
    if len(x) >= 8:
        k2, p = stats.normaltest(x)
        print("D'Agostino K2:", k2, "p-value:", p)
    jb_stat, jb_p, _, _ = jarque_bera(x)
    print("Jarque-Bera:", jb_stat, "p-value:", jb_p)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(x, bins=40, density=True, alpha=0.75)
    axes[0].set_title(f"{title}: histogram")
    stats.probplot(x, dist="norm", plot=axes[1])
    axes[1].set_title(f"{title}: normal Q-Q")
    plt.tight_layout()
    plt.show()

distribution_report(solar_daily, "Raw daily Solar")

### 3.4 Transformations: log, Box-Cox and Yeo-Johnson

- Log: requires strictly positive values.
- Box-Cox: requires strictly positive values and estimates a power parameter.
- Yeo-Johnson: accepts zero and negative values.

Fit transformation parameters on training data only in a predictive workflow. Here, the full daily series is used only for descriptive distribution comparison.

In [ ]:
x = solar_daily.dropna().to_numpy()
transformations = {}

if np.all(x > 0):
    transformations["log"] = np.log(x)
    boxcox_x, boxcox_lambda = stats.boxcox(x)
    transformations["boxcox"] = boxcox_x
    print("Box-Cox lambda:", boxcox_lambda)

pt = PowerTransformer(method="yeo-johnson", standardize=False)
transformations["yeo_johnson"] = pt.fit_transform(x.reshape(-1, 1)).ravel()

def compare_transformations(original, transformed_dict):
    rows = []
    all_series = {"original": original, **transformed_dict}
    for name, values in all_series.items():
        values = pd.Series(values).dropna()
        rows.append({
            "transformation": name,
            "skewness": skew(values, bias=False),
            "excess_kurtosis": kurtosis(values, fisher=True, bias=False),
            "mean": values.mean(),
            "std": values.std()
        })
    return pd.DataFrame(rows)

comparison = compare_transformations(x, transformations)
display(comparison)

# Side-by-side histograms and Q-Q plots
for name, values in {"original": x, **transformations}.items():
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].hist(values, bins=35, density=True, alpha=0.75)
    axes[0].set_title(f"{name}: histogram")
    stats.probplot(values, dist="norm", plot=axes[1])
    axes[1].set_title(f"{name}: Q-Q")
    plt.tight_layout()
    plt.show()

### 3.5 Important: seasonality can distort the raw distribution

A full-year Solar distribution mixes winter, summer, day and night observations. This mixture is naturally non-normal. A more careful sequence is:
1. inspect and model trend/seasonality;
2. remove trend and seasonal components;
3. inspect residual distribution;
4. only then consider a transformation if residual skewness / tails remain problematic.

Do not transform simply to force a normal-looking histogram; many forecasting models do not require normally distributed predictors or target values.

In [ ]:
# Daily Solar STL: annual seasonality is difficult to estimate robustly with only two years.
# Use a 7-day period as an illustrative weekly component; for annual seasonality,
# obtain multiple years of daily data and consider period=365.
solar_daily_stl = STL(solar_daily.dropna(), period=7, robust=True).fit()
solar_daily_stl.plot()
plt.tight_layout()
plt.show()

solar_resid = solar_daily_stl.resid.dropna()
distribution_report(solar_resid, "Daily Solar STL residual")

## 4. Special method: low-price electricity, high-price selling

The description most likely refers to **pumped-storage hydropower** (抽水蓄能), rather than a technology called “dump”. In low-price / surplus periods, electricity is used to pump water uphill; when prices are high, water is released through turbines to generate electricity. It is a storage and arbitrage mechanism, not a primary energy source.

Check the real data dictionary before finalizing the label. Some datasets report pumping as negative generation/consumption, while others report only positive generation from discharge. Therefore:
- inspect whether `Pumped Storage` can be negative;
- compare its output with electricity prices if prices are available;
- test whether discharge is concentrated in high-price hours;
- do not infer arbitrage profitability from generation alone.

In [ ]:
# Locate pumped-storage subcategory in the raw table
pump = raw.loc[
    raw["subcategory"].str.contains("pumped|storage", case=False, na=False)
].copy()

if not pump.empty:
    pump_hourly = pump.groupby("timestamp")["generation_mw"].sum().sort_index()
    fig, ax = plt.subplots(figsize=(13, 3))
    ax.plot(pump_hourly.index, pump_hourly, linewidth=0.7)
    ax.set_title("Pumped Storage generation (check sign convention)")
    ax.set_ylabel("MW")
    plt.tight_layout()
    plt.show()
    print("Min / max:", pump_hourly.min(), pump_hourly.max())
else:
    print("No pumped-storage subcategory found. Check the source data dictionary.")

## 5. Thermal generation prediction

### 5.1 Define the target and predictor timing

Target:
\[
y_t = Thermal_t
\]

Allowed predictor timing:
- Wind and Solar: current time \(t\) and historical lags may be used.
- Water and Nuclear (and any other non-Wind/Solar generation categories): **lag-1 only**.
- Do not use current Thermal or any future information.
- Calendar variables are known in advance.

Because contemporaneous Wind/Solar are permitted, this setup is a nowcasting / conditional prediction task unless these values are known before the target is observed.

In [ ]:
# Target is total Thermal generation; inputs follow the prompt's availability rules.
model_df = category_hourly[
    ["Thermal", "Wind", "Solar", "Water", "Nuclear"]
].copy()

model_df = model_df.interpolate(
    method="time", limit=3, limit_area="inside"
).dropna()

y = model_df["Thermal"].copy()
display(model_df.head())
print("Model rows:", len(model_df))


### 5.2 Chronological train / validation / test split

Never randomly shuffle time-series observations. The validation set is for feature/model selection; the test set is held out until the final evaluation.

In [ ]:
n_obs = len(model_df)
train_end = int(n_obs * 0.70)
val_end = int(n_obs * 0.85)

train_idx = model_df.index[:train_end]
val_idx = model_df.index[train_end:val_end]
test_idx = model_df.index[val_end:]

print("Train:", train_idx.min(), "to", train_idx.max(), len(train_idx))
print("Validation:", val_idx.min(), "to", val_idx.max(), len(val_idx))
print("Test:", test_idx.min(), "to", test_idx.max(), len(test_idx))

### 5.3 Feature engineering: reusable single-series and interaction features

Keep the feature code compact and modular. Wind/Solar may use current values and past-only rolling windows (the current value is included in a trailing window). Water/Nuclear are restricted to lag-1. Do not create interactions using contemporaneous Water/Nuclear.

The windows below are deliberately fewer and cover short, daily and weekly horizons. Add or remove formulas by editing the lists.

A trailing rolling window includes the current timestamp (for example, `rolling(24)` uses t-23 through t). This is allowed for Wind/Solar under the stated prompt. Water and Nuclear appear only as lag-1, including in all interaction formulas.

Percentage-change features use `(x_t - x_{t-k}) / (abs(x_{t-k}) + 1)` rather than raw division. This keeps the feature finite when Solar or Wind is zero or close to zero.

In [ ]:
def add_series_features(X, s, name):
    # Single-source time-series features: current value is allowed for Wind/Solar.
    X[f"{name}_t"] = s
    for lag in [1, 3, 6, 12, 24, 168]:
        X[f"{name}_lag{lag}"] = s.shift(lag)

    for w in [6, 24, 168]:
        r = s.rolling(w, min_periods=w)
        X[f"{name}_mean{w}"] = r.mean()
        X[f"{name}_std{w}"] = r.std()
        X[f"{name}_min{w}"] = r.min()
        X[f"{name}_max{w}"] = r.max()

    for span in [12, 48]:
        X[f"{name}_ewm{span}"] = s.ewm(span=span, adjust=False).mean()

    for lag in [1, 3, 24]:
        X[f"{name}_diff{lag}"] = s.diff(lag)
        X[f"{name}_pct{lag}"] = (s - s.shift(lag)) / (s.shift(lag).abs() + 1)

    # Useful shape/ramp features
    X[f"{name}_ramp_abs"] = s.diff().abs()
    X[f"{name}_z24"] = (s - s.rolling(24, min_periods=24).mean()) / (
        s.rolling(24, min_periods=24).std() + 1e-6
    )
    return X


def make_features(data):
    X = pd.DataFrame(index=data.index)

    # A. Single-source features
    for col in ["Wind", "Solar"]:
        X = add_series_features(X, data[col], col)

    # B. Restricted sources: lag-1 only
    for col in ["Water", "Nuclear"]:
        X[f"{col}_lag1"] = data[col].shift(1)

    # C. Calendar features
    hour = data.index.hour
    dow = data.index.dayofweek
    month = data.index.month
    X["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    X["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    X["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    X["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    X["month_sin"] = np.sin(2 * np.pi * month / 12)
    X["month_cos"] = np.cos(2 * np.pi * month / 12)
    X["is_weekend"] = (dow >= 5).astype(int)

    # D. Multi-source interactions: only contemporaneous Wind/Solar are combined.
    wind, solar = data["Wind"], data["Solar"]
    X["renewable_sum"] = wind + solar
    X["renewable_diff"] = wind - solar
    X["renewable_product"] = wind * solar
    X["renewable_ratio"] = wind / (solar.abs() + 1)
    X["wind_share"] = wind / (wind + solar + 1)
    X["solar_share"] = solar / (wind + solar + 1)

    # E. Cross-series changes and relative positions
    X["wind_minus_solar_ramp"] = wind.diff() - solar.diff()
    X["renewable_ramp"] = (wind + solar).diff()
    X["renewable_mean24"] = (wind + solar).rolling(24, min_periods=24).mean()
    X["renewable_std24"] = (wind + solar).rolling(24, min_periods=24).std()

    return X.replace([np.inf, -np.inf], np.nan)


X_all = make_features(model_df)
valid = X_all.notna().all(axis=1) & y.notna()
X_all, y = X_all.loc[valid], y.loc[valid]
print("Candidate feature count:", X_all.shape[1])
display(X_all.head())

In [ ]:

# ==========================================
# Advanced Manual Feature Engineering
# Target: Other_Production
# Inputs: Wind and Hydroelectric only
# ==========================================

def make_features(data):

    X = pd.DataFrame(index=data.index)

    wind = data["Wind"]
    hydro = data["Hydroelectric"]

    # ------------------------------------------
    # 1. Current supply and historical persistence
    # ------------------------------------------

    X["Wind_current"] = wind
    X["Hydro_current"] = hydro

    X["Wind_lag1"] = wind.shift(1)
    X["Hydro_lag1"] = hydro.shift(1)

    X["Wind_lag24"] = wind.shift(24)
    X["Hydro_lag24"] = hydro.shift(24)

    X["Wind_lag168"] = wind.shift(168)
    X["Hydro_lag168"] = hydro.shift(168)

    # ------------------------------------------
    # 2. Momentum and acceleration
    # ------------------------------------------

    # Short-term changes
    wind_change = wind.diff()
    hydro_change = hydro.diff()

    X["Wind_momentum"] = wind.diff(3)
    X["Hydro_momentum"] = hydro.diff(3)

    X["Wind_daily_change"] = wind.diff(24)
    X["Hydro_daily_change"] = hydro.diff(24)

    # Acceleration: change in the rate of change
    X["Wind_acceleration"] = wind.diff().diff()
    X["Hydro_acceleration"] = hydro.diff().diff()

    # Relative changes
    X["Wind_growth"] = wind_change / (wind.shift(1).abs() + 1)
    X["Hydro_growth"] = hydro_change / (hydro.shift(1).abs() + 1)

    # ------------------------------------------
    # 3. Volatility and stability
    # ------------------------------------------

    wind_vol = wind_change.rolling(24).std()
    hydro_vol = hydro_change.rolling(24).std()

    X["Wind_volatility"] = wind_vol
    X["Hydro_volatility"] = hydro_vol

    X["Wind_weekly_vol"] = wind_change.rolling(168).std()
    X["Hydro_weekly_vol"] = hydro_change.rolling(168).std()

    # Volatility relative to recent level
    X["Wind_vol_ratio"] = wind_vol / (wind.rolling(24).mean().abs() + 1)
    X["Hydro_vol_ratio"] = hydro_vol / (hydro.rolling(24).mean().abs() + 1)

    # ------------------------------------------
    # 4. Supply shocks and abnormal movements
    # ------------------------------------------

    wind_mean = wind.rolling(168).mean()
    hydro_mean = hydro.rolling(168).mean()

    wind_std = wind.rolling(168).std()
    hydro_std = hydro.rolling(168).std()

    X["Wind_shock"] = (wind - wind_mean) / (wind_std + 1e-6)
    X["Hydro_shock"] = (hydro - hydro_mean) / (hydro_std + 1e-6)

    X["Wind_shock_abs"] = X["Wind_shock"].abs()
    X["Hydro_shock_abs"] = X["Hydro_shock"].abs()

    # ------------------------------------------
    # 5. Mean reversion
    # ------------------------------------------

    X["Wind_mean_gap"] = wind - wind.rolling(168).mean()
    X["Hydro_mean_gap"] = hydro - hydro.rolling(168).mean()

    X["Wind_mean_gap_ratio"] = (
        X["Wind_mean_gap"] / (wind.rolling(168).mean().abs() + 1)
    )

    X["Hydro_mean_gap_ratio"] = (
        X["Hydro_mean_gap"] / (hydro.rolling(168).mean().abs() + 1)
    )

    # ------------------------------------------
    # 6. Energy mix and interaction
    # ------------------------------------------

    total = wind + hydro

    X["Renewable_total"] = total
    X["Renewable_diff"] = wind - hydro

    X["Wind_share"] = wind / (total + 1)
    X["Hydro_share"] = hydro / (total + 1)

    X["Mix_imbalance"] = (
        (wind - hydro).abs() / (total.abs() + 1)
    )

    X["Wind_Hydro_product"] = wind * hydro

    # Combined supply changes
    X["Total_momentum"] = total.diff(3)
    X["Total_daily_change"] = total.diff(24)

    X["Total_volatility"] = total.diff().rolling(24).std()

    # ------------------------------------------
    # 7. Cross-source dynamics
    # ------------------------------------------

    X["Wind_Hydro_change_corr"] = (
        wind_change.rolling(168).corr(hydro_change)
    )

    X["Wind_Hydro_change_cov"] = (
        wind_change.rolling(168).cov(hydro_change)
    )

    X["Wind_Hydro_momentum_diff"] = (
        wind.diff(24) - hydro.diff(24)
    )

    # ------------------------------------------
    # 8. Regime indicators
    # ------------------------------------------

    X["Wind_high_regime"] = (
        wind > wind.rolling(168).median()
    ).astype(int)

    X["Hydro_high_regime"] = (
        hydro > hydro.rolling(168).median()
    ).astype(int)

    X["Wind_rising"] = (wind_change > 0).astype(int)
    X["Hydro_rising"] = (hydro_change > 0).astype(int)

    X["Both_rising"] = (
        (wind_change > 0) & (hydro_change > 0)
    ).astype(int)

    X["Both_falling"] = (
        (wind_change < 0) & (hydro_change < 0)
    ).astype(int)

    # ------------------------------------------
    # 9. Temporal seasonality
    # ------------------------------------------

    hour = data.index.hour
    weekday = data.index.dayofweek
    month = data.index.month
    day_year = data.index.dayofyear

    X["Hour_sin"] = np.sin(2 * np.pi * hour / 24)
    X["Hour_cos"] = np.cos(2 * np.pi * hour / 24)

    X["Weekday_sin"] = np.sin(2 * np.pi * weekday / 7)
    X["Weekday_cos"] = np.cos(2 * np.pi * weekday / 7)

    X["Month_sin"] = np.sin(2 * np.pi * month / 12)
    X["Month_cos"] = np.cos(2 * np.pi * month / 12)

    X["Year_sin"] = np.sin(2 * np.pi * day_year / 365.25)
    X["Year_cos"] = np.cos(2 * np.pi * day_year / 365.25)

    X["Is_weekend"] = (weekday >= 5).astype(int)

    # ------------------------------------------
    # 10. Clean
    # ------------------------------------------

    X = X.replace([np.inf, -np.inf], np.nan)

    return X


# Build X and y separately
X_all = make_features(model_df)
y = model_df["Other_Production"].copy()

valid = X_all.notna().all(axis=1) & y.notna()

X_all = X_all.loc[valid]
y = y.loc[valid]

print("Number of features:", X_all.shape[1])
print("Number of observations:", len(X_all))

display(X_all.head())
display(X_all.describe().T)


### 5.4 Factor screening on TRAIN only

1. Calculate each factor's correlation with the target using the training window only.
2. Rank by absolute correlation.
3. Remove highly correlated factors with each other (multicollinearity filter).
4. Keep a manageable set (for example, 40–80 features).
5. Apply exactly the same selected factor set to validation and test.

Feature ranking and redundancy filtering use only the training period. Correlation is a screening tool, not proof of predictive value; final value is judged on validation performance.

In [ ]:
X_train = X_all.loc[train_idx.intersection(X_all.index)]
y_train = y.loc[X_train.index]
X_val = X_all.loc[val_idx.intersection(X_all.index)]
y_val = y.loc[X_val.index]
X_test = X_all.loc[test_idx.intersection(X_all.index)]
y_test = y.loc[X_test.index]

# Target correlation: Pearson and Spearman
corr_table = pd.DataFrame({
    "pearson": X_train.corrwith(y_train),
    "spearman": X_train.corrwith(y_train, method="spearman")
})
corr_table["abs_pearson"] = corr_table["pearson"].abs()
display(corr_table.sort_values("abs_pearson", ascending=False).head(30))

# Feature-feature correlation heatmap for the strongest candidates
candidates = corr_table["abs_pearson"].nlargest(30).index
cm = X_train[candidates].corr()
fig, ax = plt.subplots(figsize=(11, 8))
im = ax.imshow(cm, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(candidates)), candidates, rotation=90, fontsize=7)
ax.set_yticks(range(len(candidates)), candidates, fontsize=7)
fig.colorbar(im, ax=ax, label="Correlation")
ax.set_title("Top candidate factor correlation")
plt.tight_layout()
plt.show()

def select_features(X, y, top_n=50, threshold=0.95):
    ranked = X.corrwith(y).abs().sort_values(ascending=False)
    candidates = ranked.head(top_n).index
    cm = X[candidates].corr().abs()
    selected = []
    for col in candidates:
        if all(cm.loc[col, old] < threshold for old in selected):
            selected.append(col)
    return selected

selected_features = select_features(X_train, y_train, top_n=60, threshold=0.95)
print("Selected features:", len(selected_features))
display(pd.Series(selected_features, name="selected_feature").to_frame())

Xtr, Xva, Xte = (
    X_train[selected_features],
    X_val[selected_features],
    X_test[selected_features]
)

### 5.5 Compare seven representative CPU models

Models: Ridge, ElasticNet, Random Forest, Extra Trees, HistGradientBoosting, GradientBoosting and AdaBoost. These cover regularized linear models, bagging and boosting without requiring a GPU. The validation comparison table is displayed below.

In [ ]:
def metrics(y_true, pred):
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": mean_squared_error(y_true, pred) ** 0.5,
        "R2": r2_score(y_true, pred)
    }

# CPU-friendly models; no GPU required.
models = {
    "Ridge": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10)),
    "ElasticNet": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000)),
    "RandomForest": make_pipeline(SimpleImputer(strategy="median"), RandomForestRegressor(n_estimators=150, max_depth=12, min_samples_leaf=3, max_features=0.8, n_jobs=-1, random_state=RANDOM_STATE)),
    "ExtraTrees": make_pipeline(SimpleImputer(strategy="median"), ExtraTreesRegressor(n_estimators=150, max_depth=14, min_samples_leaf=3, max_features=0.8, n_jobs=-1, random_state=RANDOM_STATE)),
    "HistGradientBoosting": make_pipeline(SimpleImputer(strategy="median"), HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=20, l2_regularization=1.0, random_state=RANDOM_STATE)),
    "GradientBoosting": make_pipeline(SimpleImputer(strategy="median"), GradientBoostingRegressor(n_estimators=150, learning_rate=0.04, max_depth=2, loss="huber", random_state=RANDOM_STATE)),
    "AdaBoost": make_pipeline(SimpleImputer(strategy="median"), AdaBoostRegressor(n_estimators=100, learning_rate=0.05, random_state=RANDOM_STATE))
}

fitted_models, validation_predictions, rows = {}, {}, []
for name, model in models.items():
    model.fit(Xtr, y_train)
    pred = model.predict(Xva)
    fitted_models[name] = model
    validation_predictions[name] = pred
    rows.append({"model": name, **metrics(y_val, pred)})

validation_results = pd.DataFrame(rows).sort_values("RMSE")
display(validation_results)

### 5.6 Small, readable parameter search

Tune only a few meaningful parameters. Keep the validation period fixed. The code below demonstrates a small Random Forest grid; expand only if runtime permits.

When tuning is enabled, each model's best validation configuration replaces its default fitted model and the validation comparison table is refreshed. Test data are not used during tuning.

### Selectively run parameter grids

Each grid is intentionally small and CPU-friendly. Set `RUN_TUNING = True` only when you want to run the tuning cells. You can comment out model blocks you do not need. Every block uses the same train/validation split and displays a sorted result table.

In [ ]:
RUN_TUNING = False  # set True to run; grids are intentionally small

def tune_model(name, estimator, grid, scale=False):
    rows, fitted = [], []
    for params in ParameterGrid(grid):
        steps = [SimpleImputer(strategy="median")]
        if scale:
            steps.append(StandardScaler())
        steps.append(estimator(**params))
        model = make_pipeline(*steps)
        model.fit(Xtr, y_train)
        pred = model.predict(Xva)
        rows.append({**params, **metrics(y_val, pred)})
        fitted.append(model)

    result = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
    display(result)
    best_params = result.iloc[0][list(grid)].to_dict()
    best_params = {
        k: (int(v) if isinstance(v, np.integer) else v)
        for k, v in best_params.items()
    }

    steps = [SimpleImputer(strategy="median")]
    if scale:
        steps.append(StandardScaler())
    steps.append(estimator(**best_params))
    best_model = make_pipeline(*steps)
    best_model.fit(Xtr, y_train)
    print(name, "best parameters:", best_params)
    return best_model, result


if RUN_TUNING:
    tuned_models = {}
    tuning_results = {}

    # Ridge
    tuned_models["Ridge"], tuning_results["Ridge"] = tune_model(
        "Ridge", Ridge, {"alpha": [0.1, 1, 10, 50, 100]}, scale=True
    )

    # ElasticNet
    tuned_models["ElasticNet"], tuning_results["ElasticNet"] = tune_model(
        "ElasticNet", ElasticNet,
        {"alpha": [0.001, 0.01, 0.1], "l1_ratio": [0.2, 0.5, 0.8], "max_iter": [5000]},
        scale=True
    )

    # Random Forest
    tuned_models["RandomForest"], tuning_results["RandomForest"] = tune_model(
        "RandomForest", RandomForestRegressor,
        {"n_estimators": [100, 200], "max_depth": [8, 12],
         "min_samples_leaf": [2, 5], "max_features": [0.7, 1.0],
         "n_jobs": [-1], "random_state": [RANDOM_STATE]}
    )

    # Extra Trees
    tuned_models["ExtraTrees"], tuning_results["ExtraTrees"] = tune_model(
        "ExtraTrees", ExtraTreesRegressor,
        {"n_estimators": [100, 200], "max_depth": [10, 16],
         "min_samples_leaf": [2, 5], "max_features": [0.7, 1.0],
         "n_jobs": [-1], "random_state": [RANDOM_STATE]}
    )

    # HistGradientBoosting
    tuned_models["HistGradientBoosting"], tuning_results["HistGradientBoosting"] = tune_model(
        "HistGradientBoosting", HistGradientBoostingRegressor,
        {"max_iter": [150, 250], "learning_rate": [0.03, 0.06],
         "max_leaf_nodes": [15, 31], "l2_regularization": [0.5, 2.0],
         "random_state": [RANDOM_STATE]}
    )

    # GradientBoosting
    tuned_models["GradientBoosting"], tuning_results["GradientBoosting"] = tune_model(
        "GradientBoosting", GradientBoostingRegressor,
        {"n_estimators": [100, 200], "learning_rate": [0.03, 0.06],
         "max_depth": [2, 3], "loss": ["huber"],
         "random_state": [RANDOM_STATE]}
    )

    # AdaBoost
    tuned_models["AdaBoost"], tuning_results["AdaBoost"] = tune_model(
        "AdaBoost", AdaBoostRegressor,
        {"n_estimators": [50, 100, 200], "learning_rate": [0.03, 0.1],
         "loss": ["linear", "square"], "random_state": [RANDOM_STATE]}
    )

    # Replace default models with tuned versions and refresh validation scores.
    for name, model in tuned_models.items():
        fitted_models[name] = model
        validation_predictions[name] = model.predict(Xva)

    validation_results = pd.DataFrame([
        {"model": name, **metrics(y_val, pred)}
        for name, pred in validation_predictions.items()
    ]).sort_values("RMSE").reset_index(drop=True)

    display(validation_results)


## automl

In [ ]:

# ==========================================
# AutoGluon: Three Feature Sets
# Target: Thermal
# ==========================================

# Install if needed:
# !pip install -U autogluon.tabular

from autogluon.tabular import TabularPredictor

# 1. Prepare three feature sets
renewable_features = [
    c for c in X_train.columns
    if c.startswith(("Wind_", "Solar_", "renewable_", "wind_", "solar_"))
    or c in ["hour_sin", "hour_cos", "dow_sin", "dow_cos",
             "month_sin", "month_cos", "is_weekend"]
]

feature_sets = {
    "Manual_All": (
        X_train, X_val, X_test
    ),
    "Manual_Selected": (
        Xtr, Xva, Xte
    ),
    "Renewable_Core": (
        X_train[renewable_features],
        X_val[renewable_features],
        X_test[renewable_features]
    )
}

# 2. Train and evaluate
ag_results = {}

for name, (X_train_ag, X_val_ag, X_test_ag) in feature_sets.items():

    print("\n" + "=" * 60)
    print("Training:", name)
    print("=" * 60)

    # Build training and validation tables
    train_data = X_train_ag.copy()
    train_data["Thermal"] = y_train.reindex(train_data.index)

    val_data = X_val_ag.copy()
    val_data["Thermal"] = y_val.reindex(val_data.index)

    train_data = train_data.dropna()
    val_data = val_data.dropna()

    # Train AutoGluon
    predictor = TabularPredictor(
        label="Thermal",
        problem_type="regression",
        eval_metric="root_mean_squared_error",
        path=f"AutogluonModels/{name}"
    ).fit(
        train_data=train_data,
        tuning_data=val_data,
        time_limit=600,
        presets="good_quality",
        num_bag_folds=0,
        num_stack_levels=0,
        dynamic_stacking=False
    )

    # Validation leaderboard
    leaderboard = predictor.leaderboard(
        val_data,
        silent=True
    )

    print("\nValidation leaderboard:")
    display(leaderboard)

    # Test prediction
    pred = predictor.predict(X_test_ag)

    # Evaluation
    result = {
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "MAE": mean_absolute_error(y_test, pred),
        "R2": r2_score(y_test, pred)
    }

    print("\nTest results:")
    display(pd.DataFrame([result], index=[name]))

    ag_results[name] = {
        **result,
        "predictor": predictor,
        "predictions": pred,
        "leaderboard": leaderboard
    }


# 3. Compare all three feature sets
ag_comparison = pd.DataFrame({
    name: {
        "RMSE": result["RMSE"],
        "MAE": result["MAE"],
        "R2": result["R2"]
    }
    for name, result in ag_results.items()
}).T.sort_values("RMSE")

print("\nFinal AutoGluon Comparison:")
display(ag_comparison)


### 5.7 Ensemble: average the strongest validation models

Select models by validation RMSE, then average their predictions. Do not choose the ensemble composition based on test performance.

The ensemble is used only if its validation RMSE beats the best single model. Test metrics are reported for comparison but never used to choose the final model.

In [ ]:
# Compare the top validation model with a simple ensemble of the top 3.
top_model_names = validation_results.head(3)["model"].tolist()
best_single_name = validation_results.iloc[0]["model"]
best_single_rmse = validation_results.iloc[0]["RMSE"]

val_ensemble = np.mean(
    [validation_predictions[name] for name in top_model_names], axis=0
)
ensemble_metrics = metrics(y_val, val_ensemble)

display(pd.concat([
    validation_results,
    pd.DataFrame([{"model": "Ensemble_top3", **ensemble_metrics}])
], ignore_index=True).sort_values("RMSE"))

# Only choose the ensemble if it improves validation RMSE.
use_ensemble = ensemble_metrics["RMSE"] < best_single_rmse
final_choice = "Ensemble_top3" if use_ensemble else best_single_name
print("Validation-selected final model:", final_choice)


### 5.8 Final test evaluation — run once after decisions are fixed

The test result is the final out-of-sample estimate. If you change features or parameters after looking at test results, the test set has become part of model selection; create a new untouched holdout for an unbiased final estimate.

In [ ]:
# Final test evaluation. Model choice was already made using validation only.
test_rows, test_predictions = [], {}
for name, model in fitted_models.items():
    pred = model.predict(Xte)
    test_predictions[name] = pred
    test_rows.append({"model": name, **metrics(y_test, pred)})

test_ensemble = np.mean(
    [test_predictions[name] for name in top_model_names], axis=0
)
test_rows.append({"model": "Ensemble_top3", **metrics(y_test, test_ensemble)})

test_results = pd.DataFrame(test_rows).sort_values("RMSE")
display(test_results)

# Use the validation-selected choice for the final prediction plot.
final_test_pred = test_ensemble if use_ensemble else test_predictions[best_single_name]
print("Final selected model:", final_choice)
print("Final selected test metrics:", metrics(y_test, final_test_pred))


### 5.9 Prediction plot and residual diagnostics

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(y_test.index, y_test, label="Actual", linewidth=1)
ax.plot(y_test.index, final_test_pred, label=final_choice, linewidth=1)
ax.set_title("Thermal generation: test-period prediction")
ax.legend()
plt.tight_layout()
plt.show()

residuals = y_test - final_test_pred
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
axes[0].hist(residuals, bins=40)
axes[0].set_title("Residual histogram")
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title("Residual Q-Q")
plot_acf(residuals.dropna(), lags=48, ax=axes[2], title="Residual ACF")
plt.tight_layout()
plt.show()

# Error by hour: reveals systematic intraday weaknesses.
error_by_hour = residuals.abs().groupby(residuals.index.hour).mean()
display(error_by_hour.rename("MAE_by_hour").to_frame())

print("Residual mean:", residuals.mean())
print("Residual std:", residuals.std())
print("Residual MAE:", residuals.abs().mean())

## 6. Interpretation checklist

Before reporting results, verify:

- [ ] Is the source unit MW or MWh per interval?
- [ ] Are timestamps local time or UTC? How is daylight-saving time handled?
- [ ] Are missing values short gaps or long outages?
- [ ] Does each category sum correctly from its subcategories?
- [ ] Is pumped-storage pumping recorded as negative consumption or omitted?
- [ ] Are Wind/Solar at time *t* genuinely available at the prediction decision time?
- [ ] Are all other generation categories restricted to lag-1?
- [ ] Were all feature selection and tuning decisions made without using test data?
- [ ] Does the model beat simple baselines (e.g. Thermal lag-1, same hour previous day)?
- [ ] Are errors concentrated in particular seasons, hours or high-ramp periods?

## 7. Simple baselines worth adding

A complex model should be compared with:
- Persistence: predict Thermal(t) using Thermal(t-1). **This is a benchmark only**; it is not an allowed predictor if the task forbids current Thermal history.
- Same-hour-yesterday: Thermal(t-24), also only if allowed by the task rules.
- Linear regression using Wind(t) and Solar(t).

If the rules prohibit all Thermal lags, do not include the first two baselines in the official competition score.

In [ ]:
# Simple benchmark using only permitted contemporaneous Wind/Solar and lag-1 Water/Nuclear.
baseline_features = ["Wind_t", "Solar_t", "Water_lag1", "Nuclear_lag1"]
baseline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Ridge(alpha=10)
)
baseline.fit(X_train[baseline_features], y_train)
baseline_pred = baseline.predict(X_test[baseline_features])
print("Allowed-input Ridge baseline:", metrics(y_test, baseline_pred))


---
**End of notebook.** Replace the synthetic CSV with the official hourly generation file, update the column mapping, confirm units/timezone and rerun from top to bottom.